# AIT303 Assignment 1 — Aspect Extraction & Topic Modeling

**Author:** [Student Name]

This notebook performs aspect extraction on scraped Best Buy Bluetooth & Wireless Speaker reviews
using three topic modeling approaches:
- **LDA** (gensim LdaMulticore) — unsupervised
- **BERTopic** (all-MiniLM-L6-v2 embeddings) — unsupervised
- **CorEx** (anchored semi-supervised) — semi-supervised

**Pipeline:** Preprocessing → SpaCy Keyphrase Extraction → LDA/BERTopic → Aspect Restructuring → CorEx

## 1. Colab Setup & Data Loading

### ⚡ Colab Instructions
If running on Google Colab:
1. Upload the `data_asg/bestbuy/` folder to your Google Drive (as `data_asg/bestbuy/`)
2. Set `COLAB = True` in the config cell below
3. Run all cells — the notebook will mount your Drive

**First-run note:** The pip install cell downloads ~200MB of packages. Sentence-transformers
model (all-MiniLM-L6-v2, 80MB) downloads on first BERTopic use. Both cache after first run.

In [ ]:
# ============================================
# CONFIGURATION
# ============================================
COLAB = True

if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/data_asg'
    BESTBUY_DIR = f'{DATA_DIR}/bestbuy'
    MODEL_DIR = f'{DATA_DIR}/models'
else:
    DATA_DIR = 'data_asg'
    BESTBUY_DIR = f'{DATA_DIR}/bestbuy'
    MODEL_DIR = 'models'

print(f"Running in {'COLAB' if COLAB else 'LOCAL'} mode")
print(f"Data directory: {DATA_DIR}")
print(f"Best Buy data:  {BESTBUY_DIR}")
print(f"Model directory: {MODEL_DIR}")

In [ ]:
# Install all required packages (Colab)
# gensim 4.4.0 cannot install on Python 3.14 - this notebook runs on Colab (Python 3.10)
!pip install gensim==4.4.0 spacy==3.8.0 bertopic==0.17.4 corextopic==1.1 pyLDAvis==3.4.1 umap-learn hdbscan
!python -m spacy download en_core_web_sm

import warnings
warnings.filterwarnings('ignore')

print("Package installation complete")

In [ ]:
# ============================================
# IMPORTS
# ============================================
import os
import re
import json
import random
import warnings
import pickle
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# NLP
import spacy

# LDA
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.models.coherencemodel import CoherenceModel

# BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

# CorEx
from corextopic import corextopic as ct

# LDA visualization
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# Scikit-learn utilities
from sklearn.feature_extraction.text import CountVectorizer as SkCountVectorizer

# Reproducibility
np.random.seed(42)
random.seed(42)

print("All imports loaded successfully")
print(f"pandas {pd.__version__}, numpy {np.__version__}, spaCy {spacy.__version__}")

## 2. Load & Preprocess Reviews

In [ ]:
# Load the consolidated reviews CSV
csv_path = f'{BESTBUY_DIR}/all_reviews.csv'
df = pd.read_csv(csv_path)

print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nUnique products: {df['product_name'].nunique()}")
print(f"Total reviews:   {len(df)}")
print(f"\nFirst 3 rows:")
df.head(3)

In [ ]:
# Load product metadata catalog
catalog_path = f'{BESTBUY_DIR}/products.json'
with open(catalog_path, 'r') as f:
    products = json.load(f)
products_df = pd.DataFrame(products)

print(f"Products in catalog: {len(products_df)}")
print(f"Columns: {list(products_df.columns)}")
products_df.head(5)

In [ ]:
# Reviews per product
review_counts = df['product_name'].value_counts()

plt.figure(figsize=(12, 5))
review_counts.head(20).plot(kind='bar')
plt.title('Reviews per Product (Top 20)')
plt.xlabel('Product')
plt.ylabel('Review Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"Min reviews per product: {review_counts.min()}")
print(f"Max reviews per product: {review_counts.max()}")
print(f"Mean reviews per product: {review_counts.mean():.1f}")
print(f"Products with >= 80 reviews: {(review_counts >= 80).sum()}")

### 2.1 Text Preprocessing — Clean Review Text

Reusing the `clean_text()` pipeline from Phase 1 (sentiment_analysis_preprocessing.ipynb):
lowercase → remove HTML → remove non-alpha → normalize whitespace (per D-24).
Product metadata is stored separately and NOT prepended to review text (per D-25).

In [ ]:
def clean_text(text):
    """Clean raw review text: lowercase, remove HTML tags, remove non-alpha characters, normalize whitespace."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
# Apply Phase 1 preprocessing to review text
df['cleaned_review'] = df['review_text'].apply(clean_text)

# Verify cleaning
print("Before vs After Cleaning (sample):")
for i in range(3):
    print(f"\n--- Sample {i+1} ---")
    print(f"BEFORE: {df['review_text'].iloc[i][:150]}")
    print(f"AFTER:  {df['cleaned_review'].iloc[i][:150]}")

print(f"\nCleaned review length stats:")
print(df['cleaned_review'].str.len().describe())

# Filter out empty reviews after cleaning
empty_count = (df['cleaned_review'].str.len() == 0).sum()
print(f"\nEmpty reviews after cleaning: {empty_count} ({empty_count/len(df)*100:.1f}%)")

In [ ]:
# Tokenize cleaned reviews for gensim LDA input
df['tokens'] = df['cleaned_review'].str.split()

# Filter short reviews (minimum 3 tokens for meaningful topic contribution)
df_valid = df[df['tokens'].apply(len) >= 3].copy()

print(f"Reviews with >= 3 tokens: {len(df_valid)} / {len(df)}")
print(f"Removed {len(df) - len(df_valid)} very short reviews")

In [ ]:
# Save preprocessed DataFrame for downstream use
preprocessed_path = f'{BESTBUY_DIR}/preprocessed_reviews.csv'
df_valid.to_csv(preprocessed_path, index=False)
print(f"Preprocessed data saved to: {preprocessed_path}")
print(f"Shape: {df_valid.shape}")